# Phần 0 — Tổng quan

Phục hồi ảnh hoa và phân loại bằng MobileNetV2. Notebook này chứng minh checkpoint, dữ liệu audit, split và quy trình; full 49 được để thành bước chủ động chạy bằng GPU.

In [1]:
from pathlib import Path
import hashlib, json, sys
import numpy as np
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / 'src').exists(), 'Hãy chạy notebook từ thư mục gốc dự án'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print({'python': sys.version.split()[0], 'root': str(ROOT)})

{'python': '3.11.16', 'root': '/home/tungduong/flower-project-run1'}


## Phần 1 — Trạng thái bàn giao

CNN đã huấn luyện; full 49 và deploy chờ người dùng chạy theo hướng dẫn.

In [2]:
meta = json.loads((ROOT/'models/model_metadata.json').read_text(encoding='utf-8'))
print({k: meta[k] for k in ['architecture','status','model_size_bytes','training_duration_seconds']})
print('FULL_49_PENDING_USER_GPU' if meta['status'] != 'FULL_RUN_COMPLETE' else 'FULL_RUN_COMPLETE')

{'architecture': 'MobileNetV2', 'status': 'FULL_RUN_COMPLETE', 'model_size_bytes': 21809629, 'training_duration_seconds': 1776.5554074998945}
FULL_RUN_COMPLETE


## Phần 2 — Audit dữ liệu

Inventory là bằng chứng audit đã lưu, nên notebook vẫn chạy khi ZIP không kèm raw dataset.

In [3]:
inventory = pd.read_csv(ROOT/'data/inventory.csv')
print({'valid_images': len(inventory), 'classes': inventory['label'].nunique()})
print(inventory['label'].value_counts().sort_index().to_dict())

{'valid_images': 3670, 'classes': 5}
{'daisy': 633, 'dandelion': 898, 'roses': 641, 'sunflowers': 699, 'tulips': 799}


## Phần 3 — Split cố định

Train/validation/test được kiểm tra không giao nhau theo cả đường dẫn và SHA-256.

In [4]:
splits = {n: pd.read_csv(ROOT/f'splits/{n}.csv') for n in ['train','validation','test']}
print({n: len(df) for n, df in splits.items()})
for a,b in [('train','validation'),('train','test'),('validation','test')]:
    assert set(splits[a].relative_path).isdisjoint(set(splits[b].relative_path))
    assert set(splits[a].sha256).isdisjoint(set(splits[b].sha256))
print('SPLIT_DISJOINT_PASS')

{'train': 2571, 'validation': 549, 'test': 550}
SPLIT_DISJOINT_PASS


## Phần 4 — Tiền xử lý

EXIF transpose → RGB → letterbox LANCZOS 224×224 → MobileNetV2 preprocess_input trong graph.

## Phần 5 — Kiến trúc CNN

MobileNetV2 dùng trọng số ImageNet, head 5 lớp; huấn luyện 15 epoch head và 10 epoch fine-tune.

In [5]:
history = pd.read_csv(ROOT/'artifacts/training/history.csv')
print({'epochs_recorded': len(history), 'outputs': meta['output_shape'], 'classes': meta['class_names']})

{'epochs_recorded': 25, 'outputs': [None, 5], 'classes': ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']}


## Phần 6 — Checksum checkpoint

Checksum được tính lại trực tiếp, không tin giá trị chép tay.

In [6]:
model_path = ROOT/'models/mobilenetv2_flowers.keras'
actual_sha = hashlib.sha256(model_path.read_bytes()).hexdigest()
print({'bytes': model_path.stat().st_size, 'sha256': actual_sha, 'matches_metadata': actual_sha == meta['model_sha256']})
assert actual_sha == meta['model_sha256']

{'bytes': 21809629, 'sha256': '54e76a3464d73672dabf823b4040ef67607c49861e2a7286ca233c138dbbc44d', 'matches_metadata': True}


## Phần 7 — Suy luận mẫu bằng model thật

Tải checkpoint trong tiến trình mới và dự đoán ảnh demo đóng gói kèm dự án.

In [7]:
try:
    import tensorflow as tf
except ModuleNotFoundError:
    tf = None
if tf is None:
    print('TENSORFLOW_NOT_INSTALLED: checkpoint/checksum hợp lệ; cài requirements để chạy suy luận')
else:
    from PIL import Image, ImageOps
    model = tf.keras.models.load_model(model_path, compile=False)
    img = ImageOps.exif_transpose(Image.open(ROOT/'assets/demo_flower.jpg')).convert('RGB')
    img = ImageOps.pad(img, (224,224), method=Image.Resampling.LANCZOS, color=(0,0,0))
    x = np.asarray(img, dtype=np.float32)[None,...]
    probs = model.predict(x, verbose=0)[0]
    print({'predicted_class': meta['class_names'][int(np.argmax(probs))], 'confidence': float(np.max(probs)), 'probability_sum': float(probs.sum())})

{'predicted_class': 'daisy', 'confidence': 0.9841811656951904, 'probability_sum': 1.0}


## Phần 8 — Năm loại suy giảm

Low light, Gaussian noise, salt-and-pepper, Gaussian blur và color cast; mỗi loại có ba mức độ.

In [8]:
from src.experiment_matrix import build_experiment_matrix
matrix = build_experiment_matrix()
print({'condition_count': len(matrix), 'unique_ids': len({c.condition_id for c in matrix})})
assert len(matrix) == 49

{'condition_count': 49, 'unique_ids': 49}


## Phần 9 — Các phép phục hồi

Gamma/CLAHE, Gaussian/bilateral/median filter, sharpening/unsharp mask và cân bằng màu RGB/HSV/LAB.

## Phần 10 — Quy tắc tuning

Chỉ validation được dùng để chọn tham số: Macro F1 giảm dần → SSIM giảm dần → latency tăng dần. Test không tham gia lựa chọn.

In [9]:
locked = json.loads((ROOT/'configs/locked_enhancement_params.json').read_text(encoding='utf-8'))
print({'selection_split': locked.get('_metadata',{}).get('selection_split'), 'quick_run': locked.get('_metadata',{}).get('quick_run'), 'parameter_groups': len(locked.get('parameters',{}))})

{'selection_split': 'validation', 'quick_run': False, 'parameter_groups': 33}


## Phần 11 — Giao thức full 49

Mỗi condition chạy trên toàn bộ 550 ảnh Test; predictions được lưu một lần rồi tái sử dụng cho metrics, thống kê và error analysis.

## Phần 12 — Cờ chạy GPU

Để mặc định False khi trình bày. Chỉ bật sau khi dataset đúng cấu trúc và TensorFlow đã nhận GPU.

In [10]:
RUN_FULL_49 = False
if RUN_FULL_49:
    import subprocess
    subprocess.run([sys.executable, 'scripts/run_full_pipeline.py', '--skip-train'], check=True)
else:
    print('SKIPPED_BY_DESIGN: xem docs/RUN_FULL_49_AND_DEPLOY.md')

SKIPPED_BY_DESIGN: xem docs/RUN_FULL_49_AND_DEPLOY.md


## Phần 13 — Đọc kết quả cuối an toàn

Notebook không nhầm smoke test với kết quả báo cáo.

In [11]:
manifest_path = ROOT/'results/final/manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    assert manifest.get('metadata',{}).get('quick_run') is False
    print({'status':'FULL_RUN_COMPLETE','conditions':manifest['condition_count'],'predictions':manifest['prediction_rows']})
else:
    print('FULL_EVALUATION_PENDING')

{'status': 'FULL_RUN_COMPLETE', 'conditions': 49, 'predictions': 26950}


## Phần 14 — Streamlit readiness

Ứng dụng khóa dự đoán nếu full params/checksum/metadata chưa đồng bộ; đây là hành vi trung thực có chủ đích.

In [12]:
from app_components.readiness import inspect_artifact_readiness
r = inspect_artifact_readiness(model_path, ROOT/'models/model_metadata.json', ROOT/'models/class_names.json', ROOT/'configs/locked_enhancement_params.json')
print({'app_ready': r['ready'], 'reasons': r['errors']})

{'app_ready': True, 'reasons': []}


## Phần 15 — Test và coherence

Chạy `pytest`, core validator và `scripts/check_consistency.py`; full gate chỉ dùng sau khi full 49 hoàn tất.

## Phần 16 — Kết luận và bước tiếp theo

Checkpoint CNN, pipeline, notebook và app đã đóng gói. Tiếp theo: chạy full 49 trên WSL2 GPU, validate, rồi deploy Streamlit theo `docs/RUN_FULL_49_AND_DEPLOY.md`.